# CrossEncoder로 후보를 다시 정렬하기

이 노트북은 공개 PDF 6건과 Ch11의 30문항을 사용합니다. 첫 셀부터 순서대로 실행합니다.


## 입력 준비

문자 단위 TF-IDF로 작은 후보 목록을 먼저 만듭니다. 리랭커는 이 목록 밖의 문서를 볼 수 없습니다.

### 후보 생성과 쌍 점수화를 구분하기

`CrossEncoder.predict([(question, passage), ...])`는 질문과 각 후보 본문을 함께 입력해 관련성 로짓을 반환합니다. 코드는 큰 점수 순으로 기존 후보를 다시 정렬하며 새 문서를 추가하지 않습니다. 로짓은 0~1 확률이 아닐 수 있으므로 같은 질문 안의 순서를 읽고, 정답이 1차 후보에 있는지 먼저 확인합니다.


In [1]:
from pathlib import Path
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "labs/part3/ch12-retrieval-tuning").exists())
data = json.loads((root / "labs/part3/ch12-retrieval-tuning/data/advanced-documents.json").read_text())
queries = json.loads((root / "labs/part3/ch12-retrieval-tuning/data/advanced-queries.json").read_text())
chunks, parents = data["chunks"], {item["doc_id"]: item for item in data["parents"]}
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
matrix = vectorizer.fit_transform([item["text"] for item in chunks])
print(f"부모 문서 {len(parents)}건 · 자식 청크 {len(chunks)}개 · 평가 질문 {len(queries)}개")

부모 문서 6건 · 자식 청크 175개 · 평가 질문 30개


## 작은 성공 예제

개발용 Q001 한 문항에서 상위 8개 청크를 만들고 순서를 확인합니다.


In [2]:
query = next(item for item in queries if item["qid"] == "Q001")
assert query["split"] == "dev"
qv = vectorizer.transform([query["question"]])
base_scores = cosine_similarity(qv, matrix)[0]
candidate_idx = np.argsort(-base_scores)[:8]
for rank, idx in enumerate(candidate_idx, 1):
    print(rank, chunks[idx]["chunk_id"], f"{base_scores[idx]:.3f}")

1 E01-P01-C01 0.421
2 E01-P02-C01 0.236
3 E02-P05-C01 0.194
4 E01-P02-C04 0.172
5 E01-P05-C01 0.098
6 E01-P02-C02 0.093
7 E01-P03-C06 0.055
8 E02-P05-C02 0.048


## 실제 CrossEncoder 추론

질문과 후보 본문을 한 쌍으로 넣어 관련성 점수를 계산합니다.


In [3]:
from sentence_transformers import CrossEncoder

model_name = "BAAI/bge-reranker-v2-m3"
reranker = CrossEncoder(model_name, max_length=256, device="cpu")
pairs = [(query["question"], chunks[idx]["text"]) for idx in candidate_idx]
rerank_scores = reranker.predict(pairs, show_progress_bar=False)
order = np.argsort(-rerank_scores)
for rank, pos in enumerate(order[:5], 1):
    idx = candidate_idx[pos]
    print(rank, chunks[idx]["chunk_id"], f"{rerank_scores[pos]:.3f}")

[사용자 폴더]/working/claude/projects/한국경제신문/labs/part3/ch12-retrieval-tuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7868.79it/s]

1 E01-P02-C04 0.311
2 E01-P02-C01 0.125
3 E01-P01-C01 0.039
4 E01-P03-C06 0.004
5 E01-P05-C01 0.001


## 후보 밖 문서는 회복되는가?

후보 수를 2개로 줄여 정답 문서가 빠지면 리랭커가 추가할 수 없는지 확인합니다.


In [4]:
small_idx = np.argsort(-base_scores)[:2]
small_parents = {chunks[idx]["parent_id"] for idx in small_idx}
gold = set(query["gold_doc_ids"])
print("후보 부모 ID:", sorted(small_parents))
print("정답 문서 포함:", bool(gold & small_parents))
print("리랭커 입력 쌍 수:", len(small_idx))

후보 부모 ID: ['E01']
정답 문서 포함: True
리랭커 입력 쌍 수: 2
